In [1]:
import os


In [2]:
%pwd

'c:\\Users\\taviv\\OneDrive\\Desktop\\Projects\\End-to-End-Chest-Cancer-Classification-Using-MLFlow-and-DVC\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\taviv\\OneDrive\\Desktop\\Projects\\End-to-End-Chest-Cancer-Classification-Using-MLFlow-and-DVC'

In [5]:

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [11]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

[2026-05-27 01:58:53,619: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-27 01:58:53,621: INFO: common: yaml file: params.yaml loaded successfully]


ValueError: yaml file is empty

In [12]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = config_file_path, params_file_path: Path = params_file_path):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [14]:
import zipfile
import gdown
from cnnClassifier.utils.common import get_size
from cnnClassifier import logger

In [34]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self)->str:

        try:
            dataset_url = self.config.source_URL
            zip_download_path = self.config.local_data_file  # ✅ convert once here
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from :[{dataset_url}] into :[{zip_download_path}]")

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(url=prefix + file_id, output=str(zip_download_path), quiet=False)
            logger.info(f"Downloaded file size :[{get_size(zip_download_path)}]")

        except Exception as e:
            logger.error(f"Error occurred while downloading file: {e}")
            raise e

    def unzip_and_clean(self)->str:
        try:
            zip_download_path = self.config.local_data_file
            unzip_dir = self.config.unzip_dir
            os.makedirs(unzip_dir, exist_ok=True)
            logger.info(f"Unzipping file from :[{zip_download_path}] into :[{unzip_dir}]")
            with zipfile.ZipFile(zip_download_path, 'r') as zip_ref:
                zip_ref.extractall(unzip_dir)
            logger.info(f"Unzipped file size :[{get_size(unzip_dir)}]")
            
        except Exception as e:
            logger.error(f"Error occurred while unzipping file: {e}")
            raise e

In [35]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.unzip_and_clean()
except Exception as e:
    logger.error(f"Error occurred in data ingestion: {e}")
    raise e

[2026-05-27 02:22:13,960: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-27 02:22:13,963: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-27 02:22:13,969: INFO: common: created directory at: artifacts]
[2026-05-27 02:22:13,971: INFO: 2752441612: Downloading file from :[https://drive.google.com/file/d/1stEHAwwuzOtZZmIXSBKeLJGCeXDrFmbA/view?usp=sharing] into :[artifacts\data_ingestion\data.zip]]


Downloading...
From (original): https://drive.google.com/uc?id=1stEHAwwuzOtZZmIXSBKeLJGCeXDrFmbA
From (redirected): https://drive.google.com/uc?id=1stEHAwwuzOtZZmIXSBKeLJGCeXDrFmbA&confirm=t&uuid=01a22ffd-8218-444a-8b85-bba1e22d112c
To: c:\Users\taviv\OneDrive\Desktop\Projects\End-to-End-Chest-Cancer-Classification-Using-MLFlow-and-DVC\artifacts\data_ingestion\data.zip
100%|██████████| 49.0M/49.0M [00:04<00:00, 11.2MB/s]

[2026-05-27 02:22:21,497: INFO: 2752441612: Downloaded file size :[~ 47811 KB]]
[2026-05-27 02:22:21,497: INFO: 2752441612: Unzipping file from :[artifacts\data_ingestion\data.zip] into :[artifacts\data_ingestion]]


[2026-05-27 02:22:22,017: INFO: 2752441612: Unzipped file size :[~ 0 KB]]
